# 04 Rolling CEEMDAN decomposition

This notebook implements the strict no-look-ahead rolling decomposition. For test index t, CEEMDAN is rerun on rv_series[:t]. The default RUN_STAGE is 1 so opening and running the notebook does not trigger the full 125-day experiment.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.volatility import load_rv_series, validate_split
from src.decomposition import rolling_ceemdan, select_first_k
from src.evaluation import (
    decompose_full_and_fragment,
    reconstruction_rmse,
    rolling_reconstruction,
    run_stage1,
    save_pickle,
)

RUN_STAGE = 1
TRIALS = 100
PARALLEL = False
K_VALUES = (1, 3, 5)
RESULTS_DIR = REPO_ROOT / "results"
FIG_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["axes.unicode_minus"] = False

## Stage 1: 10-day validation

In [ ]:
rv, source = load_rv_series(REPO_ROOT / "data" / "rv_series.csv", allow_network=False)
train_end_idx, n_test = validate_split(rv)

stage1_payload = run_stage1(
    rv_series=rv,
    project_root=REPO_ROOT,
    n_days=10,
    trials=TRIALS,
    parallel=PARALLEL,
)
aligned1 = stage1_payload["imfs_rolling"]
metadata1 = stage1_payload["metadata"]
print("aligned stage-1 cube:", aligned1.shape)
print("metadata distribution:", metadata1["imf_count_distribution"])

### Recorded stage-1 finding

All first 10 decompositions returned 7 components. Average decomposition time was about 5.72 seconds. The aligned cube was (10, 7, 288), and no days failed.

## Stage 2: 50-day A/B/C comparison

Set RUN_STAGE = 2 only when intentionally rerunning this stage. The saved 50-day corrected result is included in results/tables/rolling_stage2_table.csv.

In [ ]:
if RUN_STAGE >= 2:
    n_days = 50
    aligned2, metadata2 = rolling_ceemdan(
        rv,
        train_end_idx=train_end_idx,
        max_steps=n_days,
        trials=TRIALS,
        parallel=PARALLEL,
        seed=42,
        verbose=True,
    )
    rolling_predictions2 = rolling_reconstruction(metadata2, k_values=K_VALUES)
    baseline2 = decompose_full_and_fragment(
        rv,
        train_end_idx,
        n_test=n_days,
        trials=TRIALS,
        parallel=PARALLEL,
        seed=42,
    )
    truth2 = rv.values[train_end_idx:train_end_idx + n_days]
    full_slice2 = baseline2["imfs_full"][:, train_end_idx:train_end_idx + n_days]
    fragment_slice2 = baseline2["imfs_fragment_extrapolated"][:, :n_days]
    rows = []
    full_predictions2 = {}
    fragment_predictions2 = {}
    for k in K_VALUES:
        full_predictions2[k] = np.sum(select_first_k(full_slice2, k), axis=0)
        fragment_predictions2[k] = np.sum(select_first_k(fragment_slice2, k), axis=0)
        rows.append(
            {
                "k": k,
                "full_sample_leaky_rmse": reconstruction_rmse(truth2, full_predictions2[k]),
                "fragment_arima_rmse": reconstruction_rmse(truth2, fragment_predictions2[k]),
                "rolling_strict_rmse": reconstruction_rmse(truth2, rolling_predictions2[k]),
            }
        )
    table2 = pd.DataFrame(rows)
    display(table2)
    payload2 = {
        "stage": 2,
        "n_days": n_days,
        "rv_series": rv,
        "train_end_idx": train_end_idx,
        "comparison": table2,
        "imfs_rolling": aligned2,
        "rolling_metadata": metadata2,
        "rolling_predictions": rolling_predictions2,
        "baseline": baseline2,
    }
    save_pickle(payload2, RESULTS_DIR / "rolling_stage2.pkl")
    table2.to_csv(TABLE_DIR / "rolling_stage2_table.csv", index=False)

### Recorded stage-2 finding

The corrected table uses centered and scaled ARIMA. k=1 rolling RMSE was close to fragment+ARIMA; k=3 was 24.7% lower, and k=5 was 20.5% higher. Relative to the full-sample leaky decomposition, rolling was worse by 7.5%, 33.2%, and 37.2% for k=1,3,5.

## Stage 3: full 125-day run

Stage 3 has not been executed in the packaged results. Set RUN_STAGE = 3 only after explicit confirmation.

In [ ]:
if RUN_STAGE >= 3:
    n_days = n_test
    aligned3, metadata3 = rolling_ceemdan(
        rv,
        train_end_idx=train_end_idx,
        max_steps=n_days,
        trials=TRIALS,
        parallel=PARALLEL,
        seed=42,
        verbose=True,
    )
    rolling_predictions3 = rolling_reconstruction(metadata3, k_values=K_VALUES)
    baseline3 = decompose_full_and_fragment(
        rv,
        train_end_idx,
        n_test=n_days,
        trials=TRIALS,
        parallel=PARALLEL,
        seed=42,
    )
    truth3 = rv.values[train_end_idx:train_end_idx + n_days]
    full_slice3 = baseline3["imfs_full"][:, train_end_idx:train_end_idx + n_days]
    fragment_slice3 = baseline3["imfs_fragment_extrapolated"][:, :n_days]
    rows = []
    for k in K_VALUES:
        full_pred = np.sum(select_first_k(full_slice3, k), axis=0)
        fragment_pred = np.sum(select_first_k(fragment_slice3, k), axis=0)
        rows.append(
            {
                "k": k,
                "full_sample_leaky_rmse": reconstruction_rmse(truth3, full_pred),
                "fragment_arima_rmse": reconstruction_rmse(truth3, fragment_pred),
                "rolling_strict_rmse": reconstruction_rmse(truth3, rolling_predictions3[k]),
            }
        )
    table3 = pd.DataFrame(rows)
    display(table3)
    payload3 = {
        "stage": 3,
        "n_days": n_days,
        "rv_series": rv,
        "train_end_idx": train_end_idx,
        "comparison": table3,
        "imfs_rolling": aligned3,
        "rolling_metadata": metadata3,
        "rolling_predictions": rolling_predictions3,
        "baseline": baseline3,
    }
    save_pickle(payload3, RESULTS_DIR / "rolling_stage3.pkl")
    table3.to_csv(TABLE_DIR / "rolling_stage3_table.csv", index=False)